In [1]:
!pip install tensorflow opencv-python

import pandas as pd
import numpy as np
import cv2
import os
import requests
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model

In [2]:
housing = fetch_california_housing(as_frame=True)

df = housing.frame

# Rename target
df["price"] = df["MedHouseVal"] * 100000
df = df.drop("MedHouseVal", axis=1)

df.head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,452600.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,358500.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,352100.0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,341300.0
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,342200.0


In [4]:
# Create folder
os.makedirs("images", exist_ok=True)

# Sample house image URLs
image_urls = [
    "https://images.unsplash.com/photo-1568605114967-8130f3a36994",
    "https://images.unsplash.com/photo-1572120360610-d971b9d7767c",
    "https://images.unsplash.com/photo-1600585154340-be6161a56a0c",
    "https://images.unsplash.com/photo-1599423300746-b62533397364"
]

def download_images(n):
    paths = []
    for i in range(n):
        url = image_urls[i % len(image_urls)]
        path = f"images/house_{i}.jpg"

        img_data = requests.get(url).content
        with open(path, 'wb') as f:
            f.write(img_data)

        paths.append(path)
    return paths

# Limit data (important for Colab)
df = df.sample(300, random_state=42).reset_index(drop=True)

image_paths = download_images(len(df))

In [5]:
IMG_SIZE = 224

def load_images(paths):
    imgs = []
    for path in paths:
        img = cv2.imread(path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        imgs.append(img)
    return np.array(imgs)

images = load_images(image_paths)

images = preprocess_input(images)

print(images.shape)

(300, 224, 224, 3)


In [6]:
base_model = ResNet50(weights="imagenet", include_top=False, pooling="avg")

image_features = base_model.predict(images)

print(image_features.shape)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 51s 5s/step
(300, 2048)


In [7]:
X_tab = df.drop("price", axis=1)
y = df["price"]

scaler = StandardScaler()
X_tab = scaler.fit_transform(X_tab)